### 10. Load & run interactive GSEA per group

#### 10.1 load packages and data

In [ ]:
import os
import pandas as pd
import gseapy as gp
import scanpy as sc

In [ ]:
adata_full = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad')

In [ ]:
adata_full

In [ ]:
adata_sync = sc.read("/storage/users/data/PANC/H5AD_file/adata_full_sync.h5ad")

In [ ]:
adata_sync

In [ ]:
# ─── 0.  Load the annotated expression table ──────────────────────────────────
in_path = "/storage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/expr_mean_filtered_with_cluster.csv"
df = pd.read_csv(in_path)
print(f"Loaded {df.shape[0]} genes × {df.shape[1]} cols")


In [ ]:
df

#### 10.2 build new adata object

In [ ]:
import anndata as ad
import pandas as pd
import os

df = df.rename(columns={"Unnamed: 0": "ensembl_gene_id"})
# drop duplicated UniProtID.1 and gene_symbol.1 columns
df = df.loc[:, ~df.columns.str.endswith(".1")]

# 3) Build mapping dicts
ens_list = df["ensembl_gene_id"].tolist()
ens2up   = dict(zip(df["ensembl_gene_id"], df["UniProtID"]))
ens2sym  = dict(zip(df["ensembl_gene_id"], df["gene_symbol"]))
ens2cl   = dict(zip(df["ensembl_gene_id"], df["cluster_id"]))

# 4) Subset adata_sync to only those Ensembl genes
keep = [g for g in adata_sync.var_names if g in ens_list]
adata_sub = adata_sync[:, keep].copy()

# 5) Reorder to match df order
adata_sub = adata_sub[:, ens_list].copy()

# 6) Switch var_names to UniProtID and annotate var
uni_ids = [ens2up[ens] for ens in adata_sub.var_names]
adata_sub.var_names = uni_ids
adata_sub.var["ensembl_gene_id"] = ens_list
adata_sub.var["gene_symbol"]      = [ens2sym[ens] for ens in ens_list]
adata_sub.var["cluster_id"]       = [ens2cl[ens]  for ens in ens_list]

# 7) Save the new AnnData
out_path = (
    "/storage/users/job37yv/Projects/PANC_cancer/analysis"
    "/network_module_gsea/adata_sync_modules.h5ad"
)
os.makedirs(os.path.dirname(out_path), exist_ok=True)
adata_sub.write_h5ad(out_path)

print(f"New AnnData: {adata_sub.n_obs} cells × {adata_sub.n_vars} genes")
print("Var annotation columns:", list(adata_sub.var.columns))


In [ ]:
adata_sub

#### 10. 3 perform GSEA

In [ ]:
import os
import scanpy as sc
import pandas as pd
import gseapy as gp

# ─── 1. Build UniProt‐based gene‐sets ─────────────────────────────────────────
# adata_sub.var_names are UniProt IDs, adata_sub.var["cluster_id"] holds your modules
cluster2genes = {
    f"cluster_{cid}": adata_sub.var_names[
                          adata_sub.var["cluster_id"] == cid
                      ].tolist()
    for cid in adata_sub.var["cluster_id"].unique()
}

# ─── 2. DE + prerank for each bin vs baseline ────────────────────────────────
baseline = "0_t_0.0000-0.5000"
bins     = [
    b for b in adata_sub.obs["leiden_t_bin_merged_nicer"].cat.categories
    if b != baseline
]

outdir = (
    "/storage/users/job37yv/Projects/PANC_cancer/analysis/"
    "network_module_gsea/prerank_de_results"
)
os.makedirs(outdir, exist_ok=True)

for b in bins:
    print(f"\n>>> DE & GSEA for {b} vs {baseline}")

    # 2a) Run DE (Wilcoxon)
    sc.tl.rank_genes_groups(
        adata_sub,
        groupby   = "leiden_t_bin_merged_nicer",
        groups    = [b],
        reference = baseline,
        method    = "wilcoxon"
    )

    # 2b) Extract DE results
    de_df = sc.get.rank_genes_groups_df(adata_sub, group=b)
    de_df = de_df.dropna(subset=["logfoldchanges", "names"])
    de_df = de_df[~de_df["names"].duplicated()]

    # 2c) Build prerank: index=UniProtID, value=log2FC
    rnk = pd.Series(
        de_df["logfoldchanges"].values,
        index=de_df["names"]  # these are UniProt IDs
    ).sort_values(ascending=False)

    # 2d) Run prerank GSEA with relaxed size filters
    res = gp.prerank(
        rnk             = rnk,
        gene_sets       = cluster2genes,
        threads         = 4,
        permutation_num = 1000,
        min_size        = 3,       # allow small modules
        max_size        = 2000,    # up to whole network
        outdir          = os.path.join(outdir, b.replace("/", "_")),
        seed            = 0,
        verbose         = True
    )

    # 2e) Save results
    res.res2d.to_csv(
        os.path.join(outdir, f"gsea_de_prerank_{b}.csv")
    )
    print(f"  • {res.res2d.shape[0]} enriched modules saved.")

print("\nAll done!")


In [ ]:
adata_sub

#### 10.3 visualize results

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Locate prerank result files
prerank_dir = "/storage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/prerank_de_results"
files = sorted(f for f in os.listdir(prerank_dir) if f.startswith("gsea_de_prerank_") and f.endswith(".csv"))

# 2. Read NES from each file
nes_data = {}
for fname in files:
    bin_name = fname.replace("gsea_de_prerank_", "").replace(".csv", "")
    df = pd.read_csv(os.path.join(prerank_dir, fname), index_col=0)
    if "NES" in df.columns:
        nes_data[bin_name] = df["NES"]

# 3. Build matrix (Time bins × Modules)
nes_df = pd.DataFrame(nes_data).T  # rows=time bins, cols=module names
nes_df = nes_df.sort_index()

# 4. Sort modules by overall |NES| (mean absolute NES)
module_order = nes_df.abs().mean().sort_values(ascending=False).index.tolist()
nes_df = nes_df[module_order]

# 5. Prepare data for heatmap
data = nes_df.values.astype(float)
vlim = np.nanmax(np.abs(data))
masked = np.ma.masked_invalid(data)

# 6. Plot refined heatmap
plt.figure(figsize=(12, 6))
cmap = plt.cm.coolwarm
im = plt.imshow(masked, aspect="auto", cmap=cmap, vmin=-vlim, vmax=vlim)
plt.colorbar(im, label="NES")

# Ticks
plt.yticks(np.arange(len(nes_df.index)), nes_df.index, fontsize=8)
plt.xticks(np.arange(len(nes_df.columns)), nes_df.columns, rotation=90, fontsize=6)

# Gridlines for readability
plt.grid(False)
plt.title("GSEA NES Across Modules and Time Bins", fontsize=12)
plt.xlabel("Module", fontsize=10)
plt.ylabel("Time Bin", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from matplotlib.gridspec import GridSpec

# 1. Locate prerank DE result files
prerank_dir = "/storage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/prerank_de_results"
files = sorted(f for f in os.listdir(prerank_dir)
               if f.startswith("gsea_de_prerank_") and f.endswith(".csv"))

# 2. Read NES from each file
nes_data = {}
for fname in files:
    bin_name = fname.replace("gsea_de_prerank_", "").replace(".csv", "")
    df = pd.read_csv(os.path.join(prerank_dir, fname), index_col=0)
    if "NES" in df.columns:
        nes_data[bin_name] = df["NES"]

# 3. Build matrix (Time bins × Modules)
nes_df = pd.DataFrame(nes_data).T
nes_df = nes_df.sort_index()

# 4. Compute linkage for rows (bins) and columns (modules)
row_linkage = sch.linkage(nes_df.values, method='average', metric='correlation')
col_linkage = sch.linkage(nes_df.values.T, method='average', metric='correlation')

# 5. Extract ordering from the dendrograms
row_order = sch.dendrogram(row_linkage, no_plot=True)['leaves']
col_order = sch.dendrogram(col_linkage, no_plot=True)['leaves']

# 6. Reorder the data
data        = nes_df.values[np.ix_(row_order, col_order)]
row_labels  = nes_df.index[row_order].tolist()
col_labels  = nes_df.columns[col_order].tolist()

# 7. Create figure with GridSpec
fig = plt.figure(figsize=(10, 8))
gs  = GridSpec(2, 2, width_ratios=[1,4], height_ratios=[1,4],
               wspace=0.05, hspace=0.05)

ax_col = fig.add_subplot(gs[0,1])
ax_row = fig.add_subplot(gs[1,0])
ax_heat = fig.add_subplot(gs[1,1])

# Column dendrogram
sch.dendrogram(col_linkage, ax=ax_col, orientation='top', no_labels=True)
ax_col.axis('off')

# Row dendrogram
sch.dendrogram(row_linkage, ax=ax_row, orientation='right', no_labels=True)
ax_row.axis('off')

# Heatmap
im = ax_heat.imshow(data, aspect='auto', origin='lower')
ax_heat.set_xticks(np.arange(len(col_labels)))
ax_heat.set_xticklabels(col_labels, rotation=90, fontsize=6)
ax_heat.set_yticks(np.arange(len(row_labels)))
ax_heat.set_yticklabels(row_labels, fontsize=8)

# Colorbar
cbar = fig.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)
cbar.set_label("NES")

# Labels & Title
ax_heat.set_xlabel("Module")
ax_heat.set_ylabel("Time Bin")
fig.suptitle("Clustered GSEA NES Across Modules and Time Bins", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
df

In [ ]:
g_nx

# Trash

## GSEA iterative

In [ ]:
import os
import pandas as pd
import gseapy as gp


# ─── 1.  Build gene‐sets: cluster_id → list of UniProtIDs ─────────────────────
# Use string keys like "cluster_5" so gseapy is happy
cluster2genes = {
    f"cluster_{cid}": genes
    for cid, genes in df.groupby("cluster_id")["UniProtID"]
                   .apply(list)
                   .items()
}
print(f"{len(cluster2genes)} clusters → gene‐sets")

# (Optional) write a GMT file
gmt = os.path.splitext(in_path)[0] + ".gmt"
with open(gmt, "w") as f:
    for name, genes in cluster2genes.items():
        line = [name, name] + genes
        f.write("\t".join(line) + "\n")
print("Wrote GMT to", gmt)

# ─── 2.  Identify the time‐bin columns ────────────────────────────────────────
time_bins = [c for c in df.columns if c not in ("UniProtID","gene_symbol","cluster_id")]
print("Bins:", time_bins)

# ─── 3.  Run prerank GSEA per bin ─────────────────────────────────────────────
outdir = os.path.join(os.path.dirname(in_path), "prerank_results")
os.makedirs(outdir, exist_ok=True)

for bin_name in time_bins:
    print(f"\n→ GSEA for {bin_name}")
    # 3a) ranking: UniProtID → mean expression
    rank = df.set_index("UniProtID")[bin_name].sort_values(ascending=False)

    prerank_res = gp.prerank(
        rnk            = rank,
        gene_sets      = cluster2genes,
        threads        = 40,
        permutation_num= 1000,
        outdir         = os.path.join(outdir, bin_name.replace("/", "_")),
        seed           = 42
    )

    # 3b) save results
    res_df = prerank_res.res2d
    csv_path = os.path.join(outdir, f"gsea_prerank_{bin_name}.csv")
    res_df.to_csv(csv_path)
    print(f"  • {res_df.shape[0]} terms → {csv_path}")
